In [1]:
import torch 
import numpy as np 
import h5py
import os
from pathlib import Path
import importlib
import IPython.display as ipd

import src.spatial_attn_lightning as binaural_lightning 
importlib.reload(binaural_lightning)
import yaml
from pytorch_lightning import Trainer




os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
config_path = "config/binaural_attn/markitecture.yml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

config['num_workers'] = 0
config['hparas']['batch_size'] = 2
config['audio']['rep_kwargs']['rep_on_gpu'] = True

In [14]:
config

{'corpus': {'name': 'spatialized_commonvoice_gise_scenes',
  'cue_type': 'voice_and_location',
  'task': 'word',
  'root': '/om/scratch/Fri/imgriff/datasets/spatial_audio_pipeline/assets/dataset_binaural_attn/v02',
  'with_cue_free': False},
 'audio': {'rep_type': 'cochlea_filt',
  'rep_kwargs': {'sr': 50000,
   'env_sr': 10000,
   'n_channels': 40,
   'low_lim': 40,
   'use_pad': True,
   'binaural': True,
   'rep_on_gpu': True,
   'center_crop': True,
   'out_dur': 2,
   'impulse_len': 0.25,
   'env_extraction_type': 'Half-wave Rectification',
   'downsampling_type': 'TorchTransformsResample',
   'downsampling_kwargs': {'lowpass_filter_width': 64,
    'rolloff': 0.9475937167399596,
    'resampling_method': 'kaiser_window',
    'beta': 14.769656459379492}},
  'compression_type': 'coch_p3',
  'compression_kwargs': {'scale': 1,
   'offset': 1e-07,
   'clip_value': 5,
   'power': 0.3}},
 'val_metric': 'ACC/val_acc',
 'model_name': 'binaural_attn_cue_voice_and_loc',
 'noise_kwargs': {'low

In [15]:
import pickle

class_dict = pickle.load( open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_800_word_label_to_int_dict.pkl", "rb" ))

In [6]:
ix_to_word = {v:k for k,v in class_dict.items()}

In [3]:
module = binaural_lightning.BinauralAttentionModule(config)

num_classes={'num_words': 800}
Model performing word task
cochlea_filt {'sr': 50000, 'env_sr': 10000, 'n_channels': 40, 'low_lim': 40, 'use_pad': True, 'binaural': True, 'rep_on_gpu': True, 'center_crop': True, 'out_dur': 2, 'impulse_len': 0.25, 'env_extraction_type': 'Half-wave Rectification', 'downsampling_type': 'TorchTransformsResample', 'downsampling_kwargs': {'lowpass_filter_width': 64, 'rolloff': 0.9475937167399596, 'resampling_method': 'kaiser_window', 'beta': 14.769656459379492}} coch_p3 {'scale': 1, 'offset': 1e-07, 'clip_value': 5, 'power': 0.3}
center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram


In [29]:
dataset = module.dataset(**config['corpus'], modee='train')

1068 files in train concat dataset


In [4]:
trainer = Trainer(
    precision=32,
    # default_root_dir='test_log_dump/',
    # val_check_interval=1000,
#     limit_train_batches=0.,
    limit_val_batches=0.0,
    num_nodes=1,
    gpus=1,
    # detect_anomaly=True,
    accelerator="gpu",
)
trainer.fit(module)

/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/pytorch_lightning/loops/utilities.py:91: PossibleUserWarning: `max_epochs` was not set. Setting it to 1000 epochs. To train without an epoch limit, set `max_epochs=-1`.
  rank_zero_warn(
GPU available: True, used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name             | Type                   | Params
------------------------------------------------------------
0 | audio_transforms | AudioCompose           | 0     
1 | model            | AttnSequentialAttacker | 78.7 M
2 | loss_fn          | CrossEntropyLoss       | 0     
3 | train_acc        | Accuracy               | 0     
4 | valid_acc        | Accuracy               | 0     
5 | test_acc         | Accuracy               | 0     
6 | test_confusion   | Accuracy               | 0     
--------------------------------------------

1068 files in train concat dataset


/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:240: PossibleUserWarning: The dataloader, train_dataloader, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 192 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.
  rank_zero_warn(


len training set = 3206467
Epoch 0:   0%|          | 5/1603234 [00:08<801:09:02,  1.80s/it, loss=6.68, v_num=3.11e+7] 

/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/pytorch_lightning/trainer/trainer.py:726: UserWarning: Detected KeyboardInterrupt, attempting graceful shutdown...
  rank_zero_warn("Detected KeyboardInterrupt, attempting graceful shutdown...")
